# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamzanPUCIT/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

---
**Methods: Logistic Regression and Random Forest, used as ranking scores** — plus two references:
a dummy (random order at the base rate) and my **frozen week-4 rule** (`baseline_score`, same formula,
nothing refit).

Why these two:
- My lane is a *"which pages first?"* question, so any classifier's **probability** becomes the score,
  and I judge it by **precision@50 / @100** — the same metric as week 4.
- **Logistic Regression** is the readable step up from my hand weights: it learns the weights of the
  same kind of signals and I can read the coefficients.
- **Random Forest** can learn interactions by itself — week 2 showed the CTR effect is strong for young
  pages and almost gone for 365+ pages, which a straight-line model can't express.
  `min_samples_leaf=20` keeps it from memorizing single pages.
- I did not go to gradient boosting: on this data size it adds tuning work, and the week-4 rule is already
  hard to beat. Complexity has to earn its place.

**Inputs are the same safe set as week 4** — only the earlier window (days 31–60) and static page
properties. Missing keyword / word-count values get `has_` flags instead of a silent fill (missingness
follows `content_type`). The assert in the cell below blocks every label-window or 90-day-overlap column.

In [1]:
# Setup: load the starter dataset (works in a local clone and in Colab)
from pathlib import Path
import numpy as np
import pandas as pd

RAW_URL = ("https://raw.githubusercontent.com/RamzanPUCIT/flyrank-ml-internship/"
           "main/data/raw/content_refresh_anonymized.csv")
local = [p for p in [Path("../../data/raw/content_refresh_anonymized.csv"),
                     Path("data/raw/content_refresh_anonymized.csv")] if p.exists()]
df = pd.read_csv(local[0] if local else RAW_URL)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("rows x cols:", df.shape)

# My lane slice (same definition in every notebook)
MIN_PREV_IMPR = 100          # page needs real demand in the earlier window
lane = df[df["impressions_prev_30d"] >= MIN_PREV_IMPR].copy()
# proxy target: observed later-window outcome (last 30d vs the 30d before)
lane["demand_drop"] = (lane["impressions_last_30d"] < 0.8 * lane["impressions_prev_30d"]).astype(int)

import json, sklearn
from sklearn.model_selection import GroupKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
SEED = 42
print("pandas", pd.__version__, "| sklearn", sklearn.__version__, "| seed", SEED)

# Feature frame: earlier window (days 31-60) + static page properties ONLY
lane["ctr_prev"] = lane["clicks_prev_30d"] / lane["impressions_prev_30d"] * 100
lane["log_impr_prev"] = np.log1p(lane["impressions_prev_30d"])
lane["log_clicks_prev"] = np.log1p(lane["clicks_prev_30d"])
lane["log_sessions_prev"] = np.log1p(lane["sessions_prev_30d"])
lane["has_keyword_data"] = lane["search_volume"].notna().astype(int)   # missingness follows content_type
lane["has_word_count"] = lane["word_count"].notna().astype(int)
NUM = ["log_impr_prev", "log_clicks_prev", "ctr_prev", "log_sessions_prev", "content_age_days",
       "search_volume", "competition", "cpc", "word_count", "has_keyword_data", "has_word_count"]
CAT = ["content_type", "main_intent"]
X = pd.concat([lane[NUM].fillna(0),
               pd.get_dummies(lane[CAT].fillna("unknown"), dtype=int)], axis=1)
y = lane["demand_drop"].values
groups = lane["client_id"].values

FORBIDDEN = ["trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
             "sessions_last_30d", "impressions_90d", "clicks_90d", "ctr", "avg_position",
             "days_since_last_update", "impression_tier", "position_tier", "freshness_tier"]
assert not set(X.columns) & set(FORBIDDEN), "leaky column in X"
print("features:", X.shape[1], list(X.columns))

# Frozen week-4 baseline (same formula, no fitting, no labels)
pct = lambda s: s.rank(pct=True)
lane["baseline_score"] = (0.4 * (1 - pct(lane["content_age_days"])) + 0.4 * (1 - pct(lane["ctr_prev"]))
                          + 0.2 * pct(lane["log_impr_prev"]))

rows x cols: (30000, 44)


pandas 3.0.5 | sklearn 1.9.1 | seed 42
features: 19 ['log_impr_prev', 'log_clicks_prev', 'ctr_prev', 'log_sessions_prev', 'content_age_days', 'search_volume', 'competition', 'cpc', 'word_count', 'has_keyword_data', 'has_word_count', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_commercial', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'main_intent_unknown']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

---
**Grouped 5-fold cross-validation by `client_id` (GroupKFold).**

- The tool would be used on a client's site. The honest question is *"does the score still work on a
  client it has never seen?"* Pages of one client share templates, niche, and tracking setup, so a random
  split lets the model memorize client character and look better than it is (I measure that gap below).
- **Time direction:** the starter data is one snapshot, so a train-on-past/test-on-future split across
  calendar dates isn't possible here. Inside every row the inputs (days 31–60) come strictly before the
  outcome (days 1–30), which is the time discipline this snapshot allows.
- **Uneven folds are real:** two big clients sit alone in their own fold (5,690 and 3,352 pages), the other
  folds hold 7–12 clients. So I report every fold plus mean ± std, not one pooled number.
- Seeds are fixed (`SEED = 42`); library versions are printed in the first cell.

In [2]:
gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(X, y, groups))
for i, (tr, te) in enumerate(folds):
    print(f"fold {i}: test rows {len(te):5,d} | test clients {len(set(groups[te])):2d} | "
          f"test base rate {y[te].mean():.3f} | train clients {len(set(groups[tr]))}")
assert all(not set(groups[tr]) & set(groups[te]) for tr, te in folds), "client in both train and test!"
print("OK - no client appears in both train and test in any fold")

fold 0: test rows 5,690 | test clients  1 | test base rate 0.510 | train clients 29
fold 1: test rows 3,352 | test clients  1 | test base rate 0.634 | train clients 29
fold 2: test rows 2,990 | test clients  7 | test base rate 0.547 | train clients 23
fold 3: test rows 2,989 | test clients  9 | test base rate 0.725 | train clients 21
fold 4: test rows 2,989 | test clients 12 | test base rate 0.756 | train clients 18
OK - no client appears in both train and test in any fold


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

---
**Result: the models do not beat my rule baseline on the metric that matters.**

| method (mean of 5 client-held-out folds) | precision@50 | precision@100 | ROC-AUC |
|---|---|---|---|
| base rate / dummy | 0.62 | 0.63 | 0.50 |
| **week-4 rule baseline** | **0.81** | **0.82** | 0.60 |
| logistic regression | 0.75 | 0.77 | **0.62** |
| random forest | 0.79 | 0.79 | 0.60 |

(Exact numbers with std are in the output below and in `work/outputs/w05_model_metrics.json`.)

- The rule wins at precision@50 and @100 — the top of the list, which is what an editor reads.
- Logistic regression orders the *whole* list slightly better (best AUC), but its top 50 is worse.
- The random forest wins in the two single-client folds and loses in the multi-client folds; its fold-to-fold
  spread is bigger than the rule's. It is not a stable improvement.
- Random split vs grouped split: the forest's AUC drops from ~0.70 to ~0.60 when clients are held out.
  That ~0.1 is memorized client character — exactly why the grouped split is the honest one.

**Decision:** keep the transparent rule as the queue for now; the model is decision-support evidence that
the same few signals (age, earlier clicks/CTR) carry most of the information.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

def make_models():
    return {"logistic_regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
            "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                                    random_state=SEED, n_jobs=-1)}

rows, oof = [], {m: np.zeros(len(y)) for m in ["dummy", "baseline_rule", "logistic_regression", "random_forest"]}
for i, (tr, te) in enumerate(folds):
    yt = y[te]
    scores = {"dummy": DummyClassifier(strategy="prior").fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
                       + np.random.default_rng(SEED).random(len(te)) * 1e-9,   # tie-break = random order
              "baseline_rule": lane["baseline_score"].values[te]}
    for name, model in make_models().items():
        model.fit(X.iloc[tr], y[tr])
        scores[name] = model.predict_proba(X.iloc[te])[:, 1]
    for name, s in scores.items():
        oof[name][te] = s
        rows.append({"fold": i, "method": name, "base_rate": yt.mean(),
                     "precision@50": precision_at_k(s, yt, 50), "precision@100": precision_at_k(s, yt, 100),
                     "roc_auc": roc_auc_score(yt, s)})
res = pd.DataFrame(rows)
order = ["dummy", "baseline_rule", "logistic_regression", "random_forest"]
table = (res.groupby("method")[["base_rate", "precision@50", "precision@100", "roc_auc"]]
            .agg(["mean", "std"]).round(3).loc[order])
print("Grouped 5-fold (by client) - mean and std across folds")
display(table)
print("per-fold precision@50:")
display(res.pivot(index="fold", columns="method", values="precision@50")[order].round(2))

# Random split (NOT honest here) - only to measure how much client memorization would inflate RF
rand_auc = []
for tr, te in KFold(5, shuffle=True, random_state=SEED).split(X):
    m = make_models()["random_forest"].fit(X.iloc[tr], y[tr])
    rand_auc.append(roc_auc_score(y[te], m.predict_proba(X.iloc[te])[:, 1]))
print(f"random forest ROC-AUC  random split: {np.mean(rand_auc):.3f} | grouped split: "
      f"{res[res.method == 'random_forest'].roc_auc.mean():.3f}  <- the gap is memorized client character")

OUT = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs")
OUT.mkdir(parents=True, exist_ok=True)
receipt = {"split": "GroupKFold(5) by client_id", "seed": SEED, "sklearn": sklearn.__version__,
           "rows": int(len(y)), "base_rate": round(float(y.mean()), 4),
           "mean_by_method": {m: {k: round(float(v), 4) for k, v in
                                  res[res.method == m][["precision@50", "precision@100", "roc_auc"]].mean().items()}
                              for m in order},
           "random_split_rf_auc": round(float(np.mean(rand_auc)), 4)}
(OUT / "w05_model_metrics.json").write_text(json.dumps(receipt, indent=2))
print("wrote", OUT / "w05_model_metrics.json")

Grouped 5-fold (by client) - mean and std across folds


base_rate        precision@50        precision@100        roc_auc       
                         mean    std         mean    std          mean    std    mean    std
method                                                                                      
dummy                   0.634  0.107        0.624  0.147         0.632  0.136   0.504  0.012
baseline_rule           0.634  0.107        0.808  0.079         0.816  0.065   0.599  0.016
logistic_regression     0.634  0.107        0.752  0.118         0.772  0.074   0.617  0.016
random_forest           0.634  0.107        0.788  0.112         0.788  0.114   0.596  0.018

per-fold precision@50:


method,dummy,baseline_rule,logistic_regression,random_forest
fold,,,,
0,0.48,0.76,0.78,0.78
1,0.58,0.70,0.78,0.86
2,0.50,0.82,0.62,0.60
3,0.78,0.90,0.66,0.82
4,0.78,0.86,0.92,0.88


random forest ROC-AUC  random split: 0.705 | grouped split: 0.596  <- the gap is memorized client character
wrote ../outputs/w05_model_metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

---
**What the model leans on** (cell below): content age and earlier-window CTR / clicks — the same two
signals my rule uses. That makes sense: young pages are still settling, and pages that already fail to
earn clicks lose ground. No single feature is "too perfect" (largest AUC drop ≈ 0.04), which is what I want
to see after excluding the label-window columns — no sign of leakage.

**Where it is wrong:**
- **Older pages:** inside the 365+ bucket the forest's AUC is ~0.50 — no skill at all; for 181–365 days it's
  ~0.52. Most of the skill is just "young vs old". For older pages the snapshot does not have what drives drops.
- **Clients:** for some clients the observed drop rate is far from what the model predicts on average
  (e.g. ~0.97 observed vs ~0.62 predicted). Client-level events (a site migration, a seasonal niche) move all
  pages together and are invisible in page features.
- **Three confident mistakes (case A–C):**
  - *A and B* — ~150-day-old informational pages, ~1.5k earlier impressions, 1 click, very long articles. The
    model said ~95% risk, but impressions stayed flat. "Young + weak CTR" was the pattern, but these pages
    had already stabilized; age alone can't tell settled from still-settling.
  - *C* — 280 days, 371 impressions, 0 clicks: impressions fell ~14%, just short of my 20% line. A
    borderline label, not really a wrong call — a reminder that a hard threshold turns small noise into
    "yes/no".

(The `impressions_last_30d` column is shown in the case table only to read the outcome — it is never an
input.)

**Careful claim:** observed on 18k pages from 30 clients, the learned models are not better than a
three-signal rule at choosing the first 50–100 pages to review. Next step for the capstone: per-client
queues, a keyword-volume floor, and position information from the warehouse's daily table, where windows
can be aligned properly.

In [4]:
# 1) What does the random forest lean on? permutation importance on each held-out fold, averaged
imps = []
for i, (tr, te) in enumerate(folds):
    m = make_models()["random_forest"].fit(X.iloc[tr], y[tr])
    pi = permutation_importance(m, X.iloc[te], y[te], scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
    imps.append(pd.Series(pi.importances_mean, index=X.columns))
imp = pd.concat(imps, axis=1).mean(axis=1).sort_values(ascending=False)
print("permutation importance (drop in held-out ROC-AUC when shuffled), top 6:")
display(imp.head(6).round(4).to_frame("auc_drop"))

lr = make_models()["logistic_regression"].fit(X, y)
coef = pd.Series(lr[-1].coef_[0], index=X.columns).sort_values()
print("logistic regression coefficients (standardized), most negative / most positive:")
display(pd.concat([coef.head(3), coef.tail(3)]).round(3).to_frame("coef"))

# 2) Where is it wrong? out-of-fold errors by age bucket and by client
lane["rf_oof"] = oof["random_forest"]
lane["age_bucket"] = pd.cut(lane["content_age_days"], [89, 180, 365, 5000], labels=["90-180", "181-365", "365+"])
by_age = lane.groupby("age_bucket", observed=True).apply(
    lambda g: pd.Series({"n": len(g), "drop_rate": g.demand_drop.mean(),
                         "rf_auc": roc_auc_score(g.demand_drop, g.rf_oof)}), include_groups=False)
display(by_age.round(3))
by_client = lane.groupby("client_id").apply(
    lambda g: pd.Series({"n": len(g), "drop_rate": g.demand_drop.mean(),
                         "mean_pred": g.rf_oof.mean()}), include_groups=False)
by_client["calibration_gap"] = (by_client["mean_pred"] - by_client["drop_rate"]).abs()
big = by_client[by_client.n >= 200].sort_values("calibration_gap", ascending=False)
print(f"clients with >=200 pages: {len(big)}; worst predicted-vs-observed drop rate gaps:")
display(big.head(3).reset_index(drop=True).round(3))

# 3) Three concrete confident mistakes (high predicted risk, no drop)
wrong = lane[lane.demand_drop == 0].sort_values("rf_oof", ascending=False).head(3)
show = wrong[["content_age_days", "impressions_prev_30d", "clicks_prev_30d", "ctr_prev", "search_volume",
              "word_count", "main_intent", "impressions_last_30d", "rf_oof"]].copy()
show.index = ["case A", "case B", "case C"]
display(show.round(3))

permutation importance (drop in held-out ROC-AUC when shuffled), top 6:


,auc_drop
content_age_days,0.0369
ctr_prev,0.0257
log_clicks_prev,0.0123
log_sessions_prev,0.0115
log_impr_prev,0.0017
main_intent_transactional,0.0003


logistic regression coefficients (standardized), most negative / most positive:


,coef
log_clicks_prev,-0.571
content_age_days,-0.383
log_sessions_prev,-0.232
has_word_count,0.092
main_intent_unknown,0.103
log_impr_prev,0.465


,n,drop_rate,rf_auc
age_bucket,,,
90-180,7394.0,0.713,0.600
181-365,6404.0,0.613,0.522
365+,4212.0,0.450,0.497


clients with >=200 pages: 10; worst predicted-vs-observed drop rate gaps:


,n,drop_rate,mean_pred,calibration_gap
0,970.0,0.965,0.623,0.342
1,1686.0,0.848,0.605,0.242
2,3352.0,0.634,0.750,0.117


,content_age_days,impressions_prev_30d,clicks_prev_30d,ctr_prev,search_volume,word_count,main_intent,impressions_last_30d,rf_oof
case A,151,1457,1,0.069,0.0,4699.0,informational,1474,0.952
case B,148,1707,1,0.059,0.0,5781.0,informational,1378,0.943
case C,280,371,0,0.000,10.0,1535.0,transactional,320,0.937


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.